##### Libraries and Imports

In [2]:
from pathlib import Path
import fitz  # PyMuPDF
import pandas as pd
import numpy as np
import re
import string

In [3]:
PROJECT_ROOT = Path("..").resolve()

RAW_PDF_DIR = PROJECT_ROOT / "data" / "raw_pdfs"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LABEL_DIR = PROJECT_ROOT / "data" / "labels"
MODEL_DIR = PROJECT_ROOT / "models"

for path in [RAW_PDF_DIR, PROCESSED_DIR, LABEL_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

In [4]:
def extract_basic_page_features(pdf_path: Path):
    rows = []

    doc = fitz.open(pdf_path)

    for page_index in range(len(doc)):
        page = doc[page_index]

        text = page.get_text("text") or ""
        words = re.findall(r"\b\w+\b", text)

        char_count = len(text)
        word_count = len(words)

        printable_chars = sum(1 for c in text if c in string.printable)
        whitespace_chars = sum(1 for c in text if c.isspace())
        alphabetic_chars = sum(1 for c in text if c.isalpha())
        digit_chars = sum(1 for c in text if c.isdigit())
        symbol_chars = sum(1 for c in text if not c.isalnum() and not c.isspace())

        page_area = page.rect.width * page.rect.height

        text_blocks = page.get_text("blocks")
        images = page.get_images(full=True)

        rows.append({
            "document_name": pdf_path.name,
            "page_number": page_index + 1,

            "char_count": char_count,
            "word_count": word_count,
            "avg_word_length": np.mean([len(w) for w in words]) if words else 0,

            "printable_char_ratio": printable_chars / char_count if char_count else 0,
            "whitespace_ratio": whitespace_chars / char_count if char_count else 0,
            "alphabetic_ratio": alphabetic_chars / char_count if char_count else 0,
            "digit_ratio": digit_chars / char_count if char_count else 0,
            "symbol_ratio": symbol_chars / char_count if char_count else 0,

            "text_block_count": len(text_blocks),
            "image_count": len(images),
            "page_width": page.rect.width,
            "page_height": page.rect.height,
            "page_area": page_area,

            "chars_per_page_area": char_count / page_area if page_area else 0,
            "words_per_page_area": word_count / page_area if page_area else 0,

            "text_preview": text[:300].replace("\n", " ")
        })

    doc.close()
    return rows

In [5]:
all_rows = []

pdf_files = list(RAW_PDF_DIR.glob("*.pdf"))

for pdf_path in pdf_files:
    print(f"Processing: {pdf_path.name}")
    rows = extract_basic_page_features(pdf_path)
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)

output_path = PROCESSED_DIR / "page_features_v1.csv"
df.to_csv(output_path, index=False)

df


Processing: anime_ir_report_80.pdf
Processing: application I-765.pdf
Processing: Community areas in Chicago - Wikipedia.pdf
Processing: Community_Area_Analysis_A_C.pdf
Processing: CryptoPipeline_Manual_UPDATED.pdf
Processing: CSP544_Final_Report_1.pdf
Processing: Higgs_boson.pdf
Processing: OPT I-20 Signed.pdf
Processing: OSNA_Assignment2[1].pdf
Processing: Passport.pdf
Processing: Receipt Notice.pdf
Processing: Resume General 1.2.0.pdf


,document_name,page_number,char_count,word_count,avg_word_length,printable_char_ratio,whitespace_ratio,alphabetic_ratio,digit_ratio,symbol_ratio,text_block_count,image_count,page_width,page_height,page_area,chars_per_page_area,words_per_page_area,text_preview
0,anime_ir_report_80.pdf,1,2182,304,5.953947,0.999542,0.129239,0.807058,0.021082,0.042621,37,0,594.959961,841.919983,500908.680145,0.004356,0.000607,Anime Information Retrieval System CS 429 – In...
1,anime_ir_report_80.pdf,2,2032,244,6.102459,0.973917,0.183563,0.701772,0.028543,0.086122,28,0,594.959961,841.919983,500908.680145,0.004057,0.000487,Salton & McGill (1983): Vector Space Model and...
2,anime_ir_report_80.pdf,3,1671,227,5.744493,0.989228,0.144225,0.742071,0.027528,0.086176,34,0,594.959961,841.919983,500908.680145,0.003336,0.000453,Tokenized articles → Word2Vec training (5 epoc...
3,anime_ir_report_80.pdf,4,2090,270,5.570370,0.961244,0.154545,0.679426,0.025359,0.140670,18,0,594.959961,841.919983,500908.680145,0.004172,0.000539,Methods: - validate_query(query_text) → erro...
4,anime_ir_report_80.pdf,5,2203,318,5.449686,0.994553,0.121198,0.714934,0.055379,0.108488,37,0,594.959961,841.919983,500908.680145,0.004398,0.000635,MODE B: Inside Notebook (Minimal Parameters) C...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,Passport.pdf,1,528,84,4.523810,0.998106,0.202652,0.579545,0.140152,0.077652,28,1,612.000000,431.399994,264016.796265,0.002000,0.000318,HIRG UR|TT/ REPUBLIC OF INDIA P 34-4/Surname...
171,Passport.pdf,2,388,71,4.056338,0.997423,0.201031,0.659794,0.082474,0.056701,12,1,612.000000,428.799988,262425.592529,0.001479,0.000271,fAeT/1it y HT HH/Name af Father /Legal Guard...
172,Receipt Notice.pdf,1,4832,808,4.732673,0.998551,0.174669,0.775455,0.015935,0.033940,38,0,612.000000,792.000000,484704.000000,0.009969,0.001667,Receipt Number IOE9894814084 Case Type I765 - ...
173,Receipt Notice.pdf,2,1150,185,4.962162,1.000000,0.169565,0.758261,0.040000,0.032174,12,0,612.000000,792.000000,484704.000000,0.002373,0.000382,Receipt Number IOE9894814084 Case Type I765 - ...


In [6]:
df.shape

(175, 18)

In [7]:
df[[
    "document_name",
    "page_number",
    "char_count",
    "word_count",
    "text_block_count",
    "image_count",
    "symbol_ratio",
    "text_preview"
]].head(38)

,document_name,page_number,char_count,word_count,text_block_count,image_count,symbol_ratio,text_preview
0,anime_ir_report_80.pdf,1,2182,304,37,0,0.042621,Anime Information Retrieval System CS 429 – In...
1,anime_ir_report_80.pdf,2,2032,244,28,0,0.086122,Salton & McGill (1983): Vector Space Model and...
2,anime_ir_report_80.pdf,3,1671,227,34,0,0.086176,Tokenized articles → Word2Vec training (5 epoc...
3,anime_ir_report_80.pdf,4,2090,270,18,0,0.140670,Methods: - validate_query(query_text) → erro...
4,anime_ir_report_80.pdf,5,2203,318,37,0,0.108488,MODE B: Inside Notebook (Minimal Parameters) C...
5,anime_ir_report_80.pdf,6,2107,299,39,0,0.089701,Expansive Set (Part 2) - Submission Artifacts:...
6,anime_ir_report_80.pdf,7,2969,295,10,0,0.140788,Q01 | action | 1 | article_0003| SE...
7,anime_ir_report_80.pdf,8,2456,267,18,0,0.132736,"if __name__ == ""__main__"": ..."
8,anime_ir_report_80.pdf,9,247,32,1,0,0.117409,"except subprocess.TimeoutExpired: print(""E..."
9,anime_ir_report_80.pdf,10,4453,600,4,0,0.152706,Running Scrapy crawler from crawler.py... ----...


In [8]:
LABELS = {
    "GOOD_TEXT": 0,
    "WEAK_TEXT": 1,
    "SCANNED_IMAGE": 2,
    "CORRUPTED_TEXT": 3,
    "MIXED_CONTENT": 4,
    "EMPTY_PAGE": 5
}

In [9]:
df["label"] = ""

In [10]:
label_path = LABEL_DIR / "manual_labels_v1.csv"

df.to_csv(label_path, index=False)

print(f"Saved labeling file: {label_path}")

Saved labeling file: C:\Users\rudra\Documents\VS Code Files\DocuMindAI\data\labels\manual_labels_v1.csv


### After Manually labelling text quality

In [25]:
df["page_id"] = (
    df["document_name"].astype(str)
    + "_page_"
    + df["page_number"].astype(str)
)

In [26]:
df["ocr_required"] = ""

In [27]:
labeled_path = LABEL_DIR / "manual_labels_v1.csv"

labeled_df = pd.read_csv(labeled_path)

print("Dataset shape:", labeled_df.shape)
labeled_df.head()

Dataset shape: (175, 19)


,document_name,page_number,char_count,word_count,avg_word_length,printable_char_ratio,whitespace_ratio,alphabetic_ratio,digit_ratio,symbol_ratio,text_block_count,image_count,page_width,page_height,page_area,chars_per_page_area,words_per_page_area,text_preview,label
0,anime_ir_report_80.pdf,1,2182,304,5.953947,0.999542,0.129239,0.807058,0.021082,0.042621,37,0,594.959961,841.919983,500908.6801,0.004356,0.000607,Anime Information Retrieval System CS 429 – In...,0
1,anime_ir_report_80.pdf,2,2032,244,6.102459,0.973917,0.183563,0.701772,0.028543,0.086122,28,0,594.959961,841.919983,500908.6801,0.004057,0.000487,Salton & McGill (1983): Vector Space Model and...,0
2,anime_ir_report_80.pdf,3,1671,227,5.744493,0.989228,0.144225,0.742071,0.027528,0.086176,34,0,594.959961,841.919983,500908.6801,0.003336,0.000453,Tokenized articles → Word2Vec training (5 epoc...,0
3,anime_ir_report_80.pdf,4,2090,270,5.570370,0.961244,0.154545,0.679426,0.025359,0.140670,18,0,594.959961,841.919983,500908.6801,0.004172,0.000539,Methods: - validate_query(query_text) → erro...,0
4,anime_ir_report_80.pdf,5,2203,318,5.449686,0.994553,0.121198,0.714934,0.055379,0.108488,37,0,594.959961,841.919983,500908.6801,0.004398,0.000635,MODE B: Inside Notebook (Minimal Parameters) C...,0


In [28]:
labeled_df["label"] = labeled_df["label"].astype(str).str.strip()

OCR_MAPPING = {
    "0": "NO",      # GOOD_TEXT
    "1": "YES",     # WEAK_TEXT
    "2": "YES",     # SCANNED_IMAGE
    "3": "YES",     # CORRUPTED_TEXT
    "4": "MAYBE",   # MIXED_CONTENT
    "5": "NO"       # EMPTY_PAGE, if used later
}

labeled_df["ocr_required"] = labeled_df["label"].map(OCR_MAPPING)

labeled_df["ocr_required"].value_counts(dropna=False)

ocr_required
NO       122
YES       31
MAYBE     22
Name: count, dtype: int64

In [29]:
updated_path = LABEL_DIR / "manual_labels_v2.csv"

labeled_df.to_csv(updated_path, index=False)

print("Saved:", updated_path)

Saved: C:\Users\rudra\Documents\VS Code Files\DocuMindAI\data\labels\manual_labels_v2.csv


In [51]:
labeled_df[[
    "document_name",
    "page_number",
    "label",
    "ocr_required",
]]

,document_name,page_number,label,ocr_required
0,anime_ir_report_80.pdf,1,0,NO
1,anime_ir_report_80.pdf,2,0,NO
2,anime_ir_report_80.pdf,3,0,NO
3,anime_ir_report_80.pdf,4,0,NO
4,anime_ir_report_80.pdf,5,0,NO
...,...,...,...,...
170,Passport.pdf,1,3,YES
171,Passport.pdf,2,3,YES
172,Receipt Notice.pdf,1,1,YES
173,Receipt Notice.pdf,2,1,YES


In [52]:
labeled_df["ocr_required"].value_counts(dropna=False)

ocr_required
NO       122
YES       31
MAYBE     22
Name: count, dtype: int64

In [53]:
labeled_df["ocr_required"].eq("MAYBE").sum()

np.int64(22)